# NUS ST4253 — Applied Time Series Analysis
## A detailed, intuitive and methodical student notebook with Bokeh

This notebook develops the core ideas of applied time-series analysis **from first principles to a complete forecasting workflow**.

Rather than starting immediately with ARIMA syntax, we will first understand **why time-series data are statistically different**, then build the ideas in the order in which they naturally depend on one another.

### Main learning objectives

By the end, you should be able to:

1. distinguish time-series data from ordinary i.i.d. data;
2. identify **level, trend, seasonality, cycles, shocks and noise**;
3. understand **lagged dependence**, autocovariance, ACF and PACF;
4. explain **white noise** and why good residuals should resemble it;
5. understand **weak stationarity** and why it matters;
6. use transformations and differencing to deal with non-stationarity;
7. interpret **AR($p$), MA($q$), ARMA($p,q$), ARIMA($p,d,q$)** models;
8. understand the role of stationarity and invertibility;
9. use **exponential smoothing / Holt / Holt-Winters / ETS-style models**;
10. use **STL decomposition** to separate trend, seasonality and remainder;
11. estimate and compare candidate time-series models;
12. diagnose residual autocorrelation using ACF and the Ljung–Box test;
13. evaluate forecasts without leaking future information into training;
14. compare naïve, seasonal-naïve, ETS and SARIMA-style forecasts;
15. produce forecasts together with **prediction intervals**.

---

## How to read the notebook

Each major section follows the same learning pattern:

> **Question → intuition → mathematics → simulation → visualisation → interpretation → practical implication**

The notebook uses **Bokeh for all charts**.

The first half uses simulated data because simulation lets us control the true data-generating mechanism.  
The second half uses a real atmospheric CO₂ dataset so that the complete workflow can be practised on genuine data.

# 0. Conceptual map of the module

A useful way to organise ST4253 is:

$$
\boxed{\text{Observe}}
\rightarrow
\boxed{\text{Understand structure}}
\rightarrow
\boxed{\text{Stationarise}}
\rightarrow
\boxed{\text{Model dependence}}
\rightarrow
\boxed{\text{Estimate}}
\rightarrow
\boxed{\text{Diagnose}}
\rightarrow
\boxed{\text{Forecast}}
\rightarrow
\boxed{\text{Evaluate}}
$$

There are two broad forecasting philosophies in this notebook.

### Component-based forecasting

Models such as exponential smoothing ask:

> How are the **level, trend and seasonality** evolving?

### Correlation-based stochastic modelling

ARIMA-family models ask:

> How does the current observation depend on **past observations and past shocks**?

These are complementary viewpoints rather than competing definitions of time series.

# 1. Environment and reproducibility

Required packages:

```text
numpy
pandas
scipy
statsmodels
scikit-learn
bokeh
```

If needed, install them in a separate notebook cell with:

```python
%pip install numpy pandas scipy statsmodels scikit-learn bokeh
```

We deliberately use a fixed random seed so the simulations are reproducible.

In [1]:
import warnings
from dataclasses import dataclass
from typing import Dict, Iterable, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

import statsmodels.api as sm
from statsmodels.tsa.stattools import acf, pacf, adfuller
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.holtwinters import (
    SimpleExpSmoothing,
    Holt,
    ExponentialSmoothing,
)
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tools.sm_exceptions import ConvergenceWarning

from bokeh.io import output_notebook, show
from bokeh.layouts import column, gridplot
from bokeh.models import Band, ColumnDataSource, HoverTool, Span
from bokeh.plotting import figure

warnings.simplefilter("ignore", ConvergenceWarning)

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

output_notebook()

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("Environment ready.")

Loading BokehJS ...

Environment ready.


# 2. Reusable plotting and evaluation utilities

A recurring mistake in analytical notebooks is to duplicate plotting and metric code in every section.

Instead, we create small reusable functions. This has two benefits:

1. the statistical sections remain focused on the idea being taught;
2. changing the presentation or metric definition later requires editing only one place.

The helpers below contain **no statistical modelling logic**. They are deliberately separated from the models.

In [2]:
def make_time_plot(
    series_map: Dict[str, pd.Series],
    title: str,
    y_label: str = "Value",
    width: int = 900,
    height: int = 340,
):
    """Overlay one or more time series on a Bokeh figure."""
    p = figure(
        width=width,
        height=height,
        x_axis_type="datetime",
        title=title,
        x_axis_label="Time",
        y_axis_label=y_label,
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )

    dashes = ["solid", "dashed", "dotted", "dotdash", "dashdot"]
    for i, (name, s) in enumerate(series_map.items()):
        s = pd.Series(s).dropna()
        p.line(
            s.index,
            s.values,
            line_width=2,
            line_dash=dashes[i % len(dashes)],
            legend_label=name,
        )

    if len(series_map) > 1:
        p.legend.location = "top_left"
        p.legend.click_policy = "hide"

    return p


def make_sequence_plot(
    series_map: Dict[str, Sequence[float]],
    title: str,
    x_label: str = "Observation",
    y_label: str = "Value",
    width: int = 900,
    height: int = 320,
):
    """Plot non-datetime sequences."""
    p = figure(
        width=width,
        height=height,
        title=title,
        x_axis_label=x_label,
        y_axis_label=y_label,
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )
    dashes = ["solid", "dashed", "dotted", "dotdash", "dashdot"]

    for i, (name, values) in enumerate(series_map.items()):
        arr = np.asarray(values)
        p.line(
            np.arange(len(arr)),
            arr,
            line_width=2,
            line_dash=dashes[i % len(dashes)],
            legend_label=name,
        )

    if len(series_map) > 1:
        p.legend.location = "top_left"
        p.legend.click_policy = "hide"
    return p


def make_acf_plot(
    values: Sequence[float],
    title: str,
    nlags: int = 36,
    partial: bool = False,
    width: int = 850,
    height: int = 320,
):
    """Bokeh ACF or PACF plot with approximate 95% white-noise bounds."""
    x = pd.Series(values).dropna().astype(float).to_numpy()

    if partial:
        corr = pacf(x, nlags=nlags, method="ywm")
        label = "PACF"
    else:
        corr = acf(x, nlags=nlags, fft=True)
        label = "ACF"

    lags = np.arange(len(corr))
    bound = 1.96 / np.sqrt(len(x))

    p = figure(
        width=width,
        height=height,
        title=title,
        x_axis_label="Lag",
        y_axis_label=label,
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )

    p.segment(
        x0=lags,
        y0=np.zeros_like(corr),
        x1=lags,
        y1=corr,
        line_width=2,
    )
    p.scatter(lags, corr, size=6)

    p.add_layout(
        Span(location=bound, dimension="width", line_dash="dashed", line_width=1.5)
    )
    p.add_layout(
        Span(location=-bound, dimension="width", line_dash="dashed", line_width=1.5)
    )
    p.add_layout(
        Span(location=0, dimension="width", line_width=1)
    )
    return p


def make_histogram(values, title: str, bins: int = 30, width: int = 850, height: int = 300):
    x = pd.Series(values).dropna().astype(float).to_numpy()
    hist, edges = np.histogram(x, bins=bins)

    p = figure(
        width=width,
        height=height,
        title=title,
        x_axis_label="Value",
        y_axis_label="Count",
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )
    p.quad(
        top=hist,
        bottom=0,
        left=edges[:-1],
        right=edges[1:],
        line_alpha=0.5,
        fill_alpha=0.6,
    )
    return p


def forecast_metrics(actual, predicted) -> dict:
    """Common point-forecast metrics."""
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    error = actual - predicted

    mae = np.mean(np.abs(error))
    rmse = np.sqrt(np.mean(error**2))

    nonzero = actual != 0
    mape = np.mean(np.abs(error[nonzero] / actual[nonzero])) * 100

    return {"MAE": mae, "RMSE": rmse, "MAPE_%": mape}


def chronological_split(series: pd.Series, test_size: int) -> Tuple[pd.Series, pd.Series]:
    """Never shuffle time-series observations."""
    if test_size <= 0 or test_size >= len(series):
        raise ValueError("test_size must lie between 1 and len(series)-1")
    return series.iloc[:-test_size].copy(), series.iloc[-test_size:].copy()


print("Reusable helpers defined.")

Reusable helpers defined.


# Part I — Why time-series data are different

# 3. Four very different data-generating mechanisms

Suppose we observe a sequence $Y_1,\ldots,Y_T$.

A time plot may come from very different mechanisms:

### White noise

$$
Y_t=\varepsilon_t
$$

No systematic temporal memory.

### Deterministic trend

$$
Y_t=\beta_0+\beta_1t+\varepsilon_t
$$

The expected value changes with time.

### Seasonal process

$$
Y_t=A\sin\left(\frac{2\pi t}{s}\right)+\varepsilon_t
$$

The mean repeats every $s$ observations.

### Autoregressive process

$$
Y_t=\phi Y_{t-1}+\varepsilon_t
$$

The current value inherits information from the previous value.

These mechanisms may produce superficially similar plots over a short interval, but they imply very different forecasting behaviour.

In [40]:
n = 240
t = np.arange(n)

white_noise = rng.normal(0, 1, n)
trend = 0.03 * t + rng.normal(0, 1, n)
seasonal = 2.0 * np.sin(2 * np.pi * t / 12) + rng.normal(0, 0.6, n)

ar1 = np.zeros(n)
eps = rng.normal(0, 1, n)
phi = 0.8
for i in range(1, n):
    ar1[i] = phi * ar1[i - 1] + eps[i]

plots = [
    make_sequence_plot({"White noise": white_noise}, "A. White noise"),
    make_sequence_plot({"Trend + noise": trend}, "B. Deterministic trend"),
    make_sequence_plot({"Seasonal + noise": seasonal}, "C. Seasonal process"),
    make_sequence_plot({"AR(1)": ar1}, "D. Persistent AR(1) process"),
]

show(column(*plots))

## Interpretation

Do not reduce time-series analysis to "fit a line through the graph."

Each panel answers a different question.

- **White noise:** no exploitable serial structure should remain.
- **Trend:** time itself changes the expected level.
- **Seasonality:** the calendar position matters.
- **AR(1):** recent observations contain predictive information even without trend or seasonality.

The fourth process is particularly important because its marginal mean can be constant while observations remain strongly dependent.

That is why time-series analysis needs tools beyond ordinary regression.

# 4. Lags: the language of temporal dependence

For a time series $Y_t$:

- $Y_{t-1}$ is lag 1;
- $Y_{t-2}$ is lag 2;
- $Y_{t-12}$ is lag 12.

If monthly sales have annual seasonality, $Y_{t-12}$ may be especially informative.

A lag therefore means:

> **How far into the past are we looking?**

A simple AR(2) model is

$$
Y_t=c+\phi_1Y_{t-1}+\phi_2Y_{t-2}+\varepsilon_t.
$$

The coefficients describe how strongly the recent past propagates into the present.

In [4]:
lag_demo = pd.DataFrame({
    "Y_t": ar1,
    "Y_t_minus_1": pd.Series(ar1).shift(1),
    "Y_t_minus_2": pd.Series(ar1).shift(2),
    "Y_t_minus_12": pd.Series(ar1).shift(12),
})

display(lag_demo.head(15))

,Y_t,Y_t_minus_1,Y_t_minus_2,Y_t_minus_12
0,0.000000,NaN,NaN,NaN
1,1.128019,0.000000,NaN,NaN
2,-1.378323,1.128019,0.000000,NaN
3,-2.599297,-1.378323,1.128019,NaN
4,-3.002324,-2.599297,-1.378323,NaN
5,-0.940680,-3.002324,-2.599297,NaN
6,-0.469957,-0.940680,-3.002324,NaN
7,0.391351,-0.469957,-0.940680,NaN
8,-0.827080,0.391351,-0.469957,NaN
9,-1.781200,-0.827080,0.391351,NaN


# 5. Autocovariance and autocorrelation

For a weakly stationary process, the autocovariance at lag $k$ is

$$
\gamma_k=\operatorname{Cov}(Y_t,Y_{t-k}).
$$

The autocorrelation is

$$
\rho_k=
\frac{\gamma_k}{\gamma_0}.
$$

Because $\gamma_0=\operatorname{Var}(Y_t)$, autocorrelation is dimensionless and lies between $-1$ and $1$.

For a stationary AR(1),

$$
Y_t=\phi Y_{t-1}+\varepsilon_t,
$$

the theoretical ACF is

$$
\rho_k=\phi^k.
$$

So for $\phi=0.8$:

$$
\rho_1=0.8,\quad
\rho_2=0.64,\quad
\rho_3=0.512,\ldots
$$

The effect of the past decays geometrically.

In [41]:
show(make_acf_plot(ar1, "ACF of the simulated AR(1) process", nlags=30))

## What to notice

The ACF does **not** suddenly become zero after lag 1.

Although the data-generating equation directly refers only to $Y_{t-1}$, dependence propagates recursively:

$$
Y_t
=
\phi(\phi Y_{t-2}+\varepsilon_{t-1})+\varepsilon_t
=
\phi^2Y_{t-2}+\phi\varepsilon_{t-1}+\varepsilon_t.
$$

Hence $Y_t$ is indirectly related to older observations too.

This distinction becomes crucial when interpreting the ACF and PACF.

# 6. PACF: direct dependence after controlling intermediate lags

The **partial autocorrelation** at lag $k$ measures the relationship between $Y_t$ and $Y_{t-k}$ **after removing the linear contribution of lags $1,\ldots,k-1$**.

For an AR(1), lag 2 may have large ordinary autocorrelation because

$$
Y_{t-2}\rightarrow Y_{t-1}\rightarrow Y_t.
$$

But after controlling for $Y_{t-1}$, lag 2 should add little direct information.

That is why the PACF of an ideal AR(1) approximately **cuts off after lag 1**.

In [44]:
show(gridplot([
    [
        make_acf_plot(ar1, "AR(1): ACF", nlags=24, width=430),
        make_acf_plot(ar1, "AR(1): PACF", nlags=24, partial=True, width=430),
    ]
]))

# 7. AR versus MA signatures

Consider:

### AR($p$)

$$
Y_t=\phi_1Y_{t-1}+\cdots+\phi_pY_{t-p}+\varepsilon_t
$$

### MA($q$)

$$
Y_t=\varepsilon_t+\theta_1\varepsilon_{t-1}+\cdots+\theta_q\varepsilon_{t-q}
$$

A classical identification heuristic is:

| Process | ACF | PACF |
|---|---|---|
| AR($p$) | tails off | cuts off near $p$ |
| MA($q$) | cuts off near $q$ | tails off |
| ARMA | tails off | tails off |

This is a **heuristic**, not a mechanical law. Sampling noise, near-cancellation, seasonality and non-stationarity can blur these signatures.

In [43]:
n2 = 600
burn = 100
eps2 = rng.normal(size=n2 + burn)

# AR(2)
ar2 = np.zeros(n2 + burn)
for i in range(2, n2 + burn):
    ar2[i] = 0.65 * ar2[i - 1] - 0.25 * ar2[i - 2] + eps2[i]
ar2 = ar2[burn:]

# MA(2)
eps3 = rng.normal(size=n2 + burn + 2)
ma2 = np.array([
    eps3[i] + 0.70 * eps3[i - 1] + 0.25 * eps3[i - 2]
    for i in range(2, n2 + burn + 2)
])[burn:]

show(gridplot([
    [
        make_acf_plot(ar2, "AR(2): ACF", nlags=25, width=430),
        make_acf_plot(ar2, "AR(2): PACF", nlags=25, partial=True, width=430),
    ],
    [
        make_acf_plot(ma2, "MA(2): ACF", nlags=25, width=430),
        make_acf_plot(ma2, "MA(2): PACF", nlags=25, partial=True, width=430),
    ],
]))

# Part II — Stationarity

# 8. Why stationarity matters

Many classical time-series results assume the probabilistic mechanism is stable over time.

For **weak stationarity**:

$$
E[Y_t]=\mu
$$

does not depend on $t$,

$$
\operatorname{Var}(Y_t)=\sigma^2
$$

does not depend on $t$,

and

$$
\operatorname{Cov}(Y_t,Y_{t-k})=\gamma_k
$$

depends on the lag $k$, not on the calendar time $t$.

Stationarity does **not** mean that the observed values are flat.

A stationary AR process can fluctuate substantially.  
The requirement concerns the *probability law's first two moments*, not a perfectly horizontal graph.

# 9. Stationary AR(1) versus random walk

Compare:

### Stationary AR(1)

$$
Y_t=0.8Y_{t-1}+\varepsilon_t
$$

A shock decays as

$$
0.8^k\rightarrow0.
$$

### Random walk

$$
Y_t=Y_{t-1}+\varepsilon_t
$$

A shock is permanently accumulated into the level.

This difference is much deeper than appearance: the random walk has a **unit root**.

In [45]:
n = 400

stationary_ar = np.zeros(n)
random_walk = np.zeros(n)

e1 = rng.normal(size=n)
e2 = rng.normal(size=n)

for i in range(1, n):
    stationary_ar[i] = 0.8 * stationary_ar[i - 1] + e1[i]
    random_walk[i] = random_walk[i - 1] + e2[i]

show(column(
    make_sequence_plot({"Stationary AR(1)": stationary_ar}, "Stationary AR(1)"),
    make_sequence_plot({"Random walk": random_walk}, "Non-stationary random walk"),
))

# 10. Rolling summaries as an exploratory stationarity check

A rolling mean and rolling standard deviation do not *prove* stationarity, but they can expose obvious changes in local level or scale.

For a window of length $w$,

$$
\bar Y_t^{(w)}
=
\frac{1}{w}
\sum_{j=0}^{w-1}Y_{t-j}.
$$

If this local mean drifts substantially, a constant-mean stationary model is suspicious.

In [46]:
def rolling_summary_plot(values, title, window=40):
    s = pd.Series(values)
    mean = s.rolling(window).mean()
    std = s.rolling(window).std()

    p1 = make_sequence_plot(
        {
            "Series": s,
            f"Rolling mean ({window})": mean,
        },
        title + " — level",
    )

    p2 = make_sequence_plot(
        {f"Rolling std ({window})": std},
        title + " — local variability",
        y_label="Rolling standard deviation",
    )
    return column(p1, p2)


show(rolling_summary_plot(random_walk, "Random walk", window=40))

# 11. Augmented Dickey–Fuller test

The ADF test is widely used to investigate unit-root non-stationarity.

A simplified interpretation is:

$$
H_0:\text{unit root / non-stationary}
$$

versus

$$
H_1:\text{stationary around the specified deterministic terms}.
$$

A small p-value gives evidence **against** the unit-root null.

Important cautions:

- failure to reject $H_0$ does not prove non-stationarity;
- structural breaks can distort the test;
- the deterministic trend specification matters;
- statistical significance should be combined with plots and domain reasoning.

In [10]:
def adf_summary(values, name: str) -> dict:
    x = pd.Series(values).dropna().astype(float)
    stat, pvalue, usedlag, nobs, critical, icbest = adfuller(x, autolag="AIC")
    return {
        "series": name,
        "ADF_statistic": stat,
        "p_value": pvalue,
        "used_lags": usedlag,
        "n_obs": nobs,
        "critical_5%": critical["5%"],
    }


adf_table = pd.DataFrame([
    adf_summary(stationary_ar, "Stationary AR(1)"),
    adf_summary(random_walk, "Random walk"),
])

display(adf_table.round(4))

,series,ADF_statistic,p_value,used_lags,n_obs,critical_5%
0,Stationary AR(1),-5.0624,0.0000,2,397,-2.8688
1,Random walk,-0.6059,0.8696,0,399,-2.8688


# 12. Differencing

The first difference is

$$
\Delta Y_t=Y_t-Y_{t-1}.
$$

For a random walk,

$$
Y_t=Y_{t-1}+\varepsilon_t,
$$

so

$$
\Delta Y_t=\varepsilon_t.
$$

This is the key idea behind the **I = Integrated** part of ARIMA.

If $d$ differences are required before an ARMA model becomes appropriate, the original process is represented as ARIMA($p,d,q$).

In [47]:
rw_series = pd.Series(random_walk)
rw_diff = rw_series.diff().dropna()

show(column(
    make_sequence_plot(
        {"Random walk": rw_series},
        "Before differencing",
    ),
    make_sequence_plot(
        {"First difference": rw_diff},
        "After first differencing",
    ),
    make_acf_plot(rw_diff, "ACF after differencing the random walk", nlags=25),
))

In [12]:
display(pd.DataFrame([
    adf_summary(rw_series, "Random walk"),
    adf_summary(rw_diff, "First-differenced random walk"),
]).round(4))

,series,ADF_statistic,p_value,used_lags,n_obs,critical_5%
0,Random walk,-0.6059,0.8696,0,399,-2.8688
1,First-differenced random walk,-20.0111,0.0000,0,398,-2.8688


## Important warning: do not difference automatically

Differencing is powerful but can be overused.

Over-differencing can:

- amplify noise;
- create artificial negative autocorrelation;
- make forecasts unnecessarily unstable;
- remove meaningful long-run structure.

The goal is **not** "difference until everything looks random."

The goal is:

> transform the series enough that the remaining stochastic dependence can be represented parsimoniously.

# 13. Seasonal differencing

For seasonal period $s$:

$$
\Delta_sY_t
=
Y_t-Y_{t-s}.
$$

For monthly observations with annual seasonality:

$$
s=12.
$$

Seasonal differencing compares an observation with the corresponding observation one year earlier.

This can be much more meaningful than ordinary first differencing when the dominant non-stationarity is seasonal.

In [13]:
n = 240
t = np.arange(n)
seasonal_trend = (
    0.025 * t
    + 2.5 * np.sin(2 * np.pi * t / 12)
    + rng.normal(0, 0.45, n)
)
s = pd.Series(seasonal_trend)
seasonal_diff = s.diff(12)

show(column(
    make_sequence_plot({"Original": s}, "Trend + annual seasonality"),
    make_sequence_plot(
        {"Seasonal difference (lag 12)": seasonal_diff},
        "After seasonal differencing",
    ),
))

# Part III — Exponential smoothing and ETS intuition

# 14. Simple Exponential Smoothing

For a series with approximately constant level and no systematic trend/seasonality:

$$
\ell_t
=
\alpha Y_t+(1-\alpha)\ell_{t-1},
\qquad 0\leq\alpha\leq1.
$$

The next forecast is

$$
\hat Y_{t+1|t}=\ell_t.
$$

Repeated substitution gives

$$
\ell_t
=
\alpha Y_t+
\alpha(1-\alpha)Y_{t-1}
+
\alpha(1-\alpha)^2Y_{t-2}
+\cdots
$$

so older observations receive exponentially decreasing weights.

### Interpretation of $\alpha$

Large $\alpha$:

- reacts quickly;
- tracks recent changes;
- can chase noise.

Small $\alpha$:

- smooths aggressively;
- changes slowly;
- may lag behind a genuine shift.

In [14]:
level_series = pd.Series(
    20 + rng.normal(0, 2.0, 120),
    index=pd.date_range("2015-01-01", periods=120, freq="MS")
)

ses_fast = SimpleExpSmoothing(level_series, initialization_method="estimated").fit(
    smoothing_level=0.8,
    optimized=False,
)
ses_slow = SimpleExpSmoothing(level_series, initialization_method="estimated").fit(
    smoothing_level=0.15,
    optimized=False,
)

show(make_time_plot(
    {
        "Observed": level_series,
        "SES alpha=0.80": ses_fast.fittedvalues,
        "SES alpha=0.15": ses_slow.fittedvalues,
    },
    "Effect of the smoothing parameter alpha",
    y_label="Simulated level",
))

# 15. Holt's method: add a changing trend

When a series trends, maintaining only a level is insufficient.

Holt's method maintains:

$$
\ell_t=\text{estimated level}
$$

and

$$
b_t=\text{estimated trend}.
$$

A basic $h$-step forecast is

$$
\hat Y_{t+h|t}=\ell_t+h b_t.
$$

This explicitly separates the current level from the rate at which that level is changing.

In [48]:
dates = pd.date_range("2012-01-01", periods=120, freq="MS")
trend_data = pd.Series(
    30 + 0.35 * np.arange(120) + rng.normal(0, 2.0, 120),
    index=dates,
)

holt_fit = Holt(trend_data, initialization_method="estimated").fit(optimized=True)

show(make_time_plot(
    {
        "Observed": trend_data,
        "Holt fitted": holt_fit.fittedvalues,
    },
    "Holt trend model",
))

# 16. Holt-Winters: level + trend + seasonality

For seasonal data, we additionally maintain a seasonal state.

The forecast conceptually becomes:

$$
\text{forecast}
=
\text{level}
+
\text{projected trend}
+
\text{seasonal effect}.
$$

For additive seasonality, the size of seasonal oscillations is roughly constant.

For multiplicative seasonality, seasonal amplitude grows with the level.

This distinction should come from the data rather than being chosen arbitrarily.

In [49]:
dates = pd.date_range("2010-01-01", periods=180, freq="MS")
seasonal_pattern = 4 * np.sin(2 * np.pi * np.arange(180) / 12)
hw_data = pd.Series(
    40 + 0.18 * np.arange(180) + seasonal_pattern + rng.normal(0, 1.2, 180),
    index=dates,
)

hw_fit = ExponentialSmoothing(
    hw_data,
    trend="add",
    seasonal="add",
    seasonal_periods=12,
    initialization_method="estimated",
).fit(optimized=True)

show(make_time_plot(
    {
        "Observed": hw_data,
        "Holt-Winters fitted": hw_fit.fittedvalues,
    },
    "Additive Holt-Winters model",
))

# Part IV — Real case study: Mauna Loa atmospheric CO₂

# 17. Why this dataset?

We now move from controlled simulation to a genuine observational series.

The `statsmodels` CO₂ dataset contains atmospheric CO₂ measurements at Mauna Loa.

It is useful pedagogically because it contains:

- a long-run upward trend;
- pronounced annual seasonality;
- missing weekly observations;
- realistic noise;
- enough history for train/test forecasting experiments.

We will convert the original weekly series to **monthly averages**.  
Monthly frequency makes the annual seasonal period naturally equal to:

$$
s=12.
$$

In [50]:
raw_co2 = sm.datasets.co2.load_pandas().data["co2"].copy()

co2_monthly = (
    raw_co2
    .resample("MS")
    .mean()
    .interpolate(method="time")
)

dataset_summary = pd.DataFrame({
    "value": [
        raw_co2.index.min(),
        raw_co2.index.max(),
        len(raw_co2),
        int(raw_co2.isna().sum()),
        len(co2_monthly),
        int(co2_monthly.isna().sum()),
    ]
}, index=[
    "Raw start",
    "Raw end",
    "Raw weekly observations",
    "Raw missing observations",
    "Monthly observations after resampling",
    "Monthly missing observations after interpolation",
])

display(dataset_summary)
display(co2_monthly.head())

,value
Raw start,1958-03-29 00:00:00
Raw end,2001-12-29 00:00:00
Raw weekly observations,2284
Raw missing observations,59
Monthly observations after resampling,526
Monthly missing observations after interpolation,0


1958-03-01    316.100000
1958-04-01    317.200000
1958-05-01    317.433333
1958-06-01    316.514344
1958-07-01    315.625000
Freq: MS, Name: co2, dtype: float64

In [51]:
show(make_time_plot(
    {"Monthly CO2": co2_monthly},
    "Mauna Loa monthly atmospheric CO₂",
    y_label="CO₂",
))

## Initial EDA

Before fitting any model, answer:

1. Is there a long-run trend?
2. Is there recurring seasonality?
3. Does seasonal amplitude appear roughly constant?
4. Are there obvious structural breaks?
5. Does the variance appear to grow dramatically with the level?

### Visual inference

The series has a strong upward trend and regular annual oscillation.

Therefore fitting a stationary ARMA model directly to the raw levels would be questionable.

This motivates decomposition and/or differencing.

# 18. STL decomposition

STL decomposes a series as

$$
Y_t=T_t+S_t+R_t,
$$

where:

- $T_t$ = trend;
- $S_t$ = seasonal component;
- $R_t$ = remainder.

STL is not merely cosmetic.

It lets us ask separately:

- what is changing slowly?
- what repeats predictably?
- what remains after these structures are removed?

In [52]:
stl = STL(co2_monthly, period=12, robust=True)
stl_result = stl.fit()

show(column(
    make_time_plot({"Observed": co2_monthly}, "STL — observed CO₂", y_label="CO₂"),
    make_time_plot({"Trend": stl_result.trend}, "STL — trend", y_label="Trend"),
    make_time_plot({"Seasonal": stl_result.seasonal}, "STL — seasonal component", y_label="Seasonal effect"),
    make_time_plot({"Remainder": stl_result.resid}, "STL — remainder", y_label="Remainder"),
))

## STL interpretation

The decomposition should reveal:

### Trend

The long-term atmospheric CO₂ level rises persistently.

### Seasonality

A regular annual cycle is visible.

### Remainder

The remainder is much less structured than the original series, but "looks noisy" is not enough.  
We should still examine serial correlation.

A recurring theme in ST4253 is:

> **Always diagnose the residual/remainder rather than trusting the model label.**

In [20]:
show(gridplot([[
    make_acf_plot(stl_result.resid, "ACF of STL remainder", nlags=36, width=430),
    make_acf_plot(stl_result.resid, "PACF of STL remainder", nlags=36, partial=True, width=430),
]]))

# 19. First and seasonal differences of the real series

The raw series has both trend and seasonality.

We therefore inspect:

$$
\Delta Y_t=Y_t-Y_{t-1}
$$

and

$$
\Delta_{12}Y_t=Y_t-Y_{t-12}.
$$

A combined seasonal + ordinary difference can be written

$$
(1-B)(1-B^{12})Y_t.
$$

In [53]:
co2_diff1 = co2_monthly.diff().dropna()
co2_diff12 = co2_monthly.diff(12).dropna()
co2_diff_both = co2_monthly.diff(12).diff().dropna()

show(column(
    make_time_plot({"First difference": co2_diff1}, "CO₂ first difference"),
    make_time_plot({"Seasonal difference": co2_diff12}, "CO₂ seasonal difference (lag 12)"),
    make_time_plot({"Seasonal + first difference": co2_diff_both}, "CO₂ combined differencing"),
))

In [54]:
stationarity_table = pd.DataFrame([
    adf_summary(co2_monthly, "Raw monthly CO2"),
    adf_summary(co2_diff1, "First difference"),
    adf_summary(co2_diff12, "Seasonal difference"),
    adf_summary(co2_diff_both, "Seasonal + first difference"),
])

display(stationarity_table.round(4))

,series,ADF_statistic,p_value,used_lags,n_obs,critical_5%
0,Raw monthly CO2,2.2340,0.9989,14,511,-2.8672
1,First difference,-4.7529,0.0001,13,511,-2.8672
2,Seasonal difference,-4.2909,0.0005,13,500,-2.8673
3,Seasonal + first difference,-8.4006,0.0000,15,497,-2.8674


# 20. ACF/PACF after differencing

ARIMA identification is usually performed on a series whose major non-stationarity has been addressed.

The ACF/PACF do not "tell us the model automatically."  
Instead, they provide evidence about plausible short-memory structure.

We will inspect the combined-difference series.

In [23]:
show(gridplot([[
    make_acf_plot(co2_diff_both, "Differenced CO₂ — ACF", nlags=36, width=430),
    make_acf_plot(co2_diff_both, "Differenced CO₂ — PACF", nlags=36, partial=True, width=430),
]]))

# Part V — AR, MA, ARMA and ARIMA

# 21. AR($p$): memory of past observations

An autoregressive process is

$$
Y_t
=
c+
\phi_1Y_{t-1}
+\cdots+
\phi_pY_{t-p}
+\varepsilon_t.
$$

The word **autoregressive** literally means:

> regress the series on its own past.

For AR(1):

$$
Y_t=c+\phi Y_{t-1}+\varepsilon_t.
$$

If $|\phi|<1$, shocks decay and the process can be stationary.

If $\phi=1$:

$$
Y_t=Y_{t-1}+\varepsilon_t,
$$

which is the random walk considered earlier.

# 22. MA($q$): memory of past shocks

A moving-average process is

$$
Y_t
=
\mu+
\varepsilon_t+
\theta_1\varepsilon_{t-1}
+\cdots+
\theta_q\varepsilon_{t-q}.
$$

Do not confuse this with a rolling arithmetic mean.

The "moving-average" model is about **unobserved innovation terms**.

Interpretation:

> a surprise occurring today can influence several future observations before its effect disappears.

# 23. ARMA($p,q$)

Combine both mechanisms:

$$
Y_t
=
c+
\sum_{i=1}^{p}\phi_iY_{t-i}
+
\varepsilon_t+
\sum_{j=1}^{q}\theta_j\varepsilon_{t-j}.
$$

The AR component describes persistence in observations.

The MA component describes persistence in shocks.

ARMA models assume stationarity; therefore non-stationary real data often require differencing first.

# 24. ARIMA($p,d,q$)

Let

$$
W_t=(1-B)^dY_t.
$$

If $W_t$ follows ARMA($p,q$), then $Y_t$ is ARIMA($p,d,q$):

$$
\phi(B)(1-B)^dY_t=\theta(B)\varepsilon_t.
$$

where $B$ is the backshift operator:

$$
BY_t=Y_{t-1}.
$$

This compact equation unifies:

- differencing;
- autoregression;
- moving-average errors.

For seasonal data we can extend this to:

$$
ARIMA(p,d,q)(P,D,Q)_s.
$$

For monthly CO₂:

$$
s=12.
$$

# Part VI — Forecast experiment design

# 25. Why ordinary random train/test splitting is wrong

For ordinary tabular ML, random splitting is often sensible.

For forecasting it can leak the future into the past.

Bad idea:

```text
Train: 1960, 1980, 1993, 2001, ...
Test:  1965, 1978, 1990, ...
```

Correct chronological split:

```text
Past -----------------------------------> Future

|--------------- training ---------------|--- test ---|
```

We will reserve the final **60 months = 5 years** as the test period.

In [55]:
TEST_MONTHS = 60
train, test = chronological_split(co2_monthly, TEST_MONTHS)

print("Train:", train.index.min(), "to", train.index.max(), "n =", len(train))
print("Test :", test.index.min(), "to", test.index.max(), "n =", len(test))

show(make_time_plot(
    {"Training": train, "Held-out test": test},
    "Chronological train/test split",
    y_label="CO₂",
))

Train: 1958-03-01 00:00:00 to 1996-12-01 00:00:00 n = 466
Test : 1997-01-01 00:00:00 to 2001-12-01 00:00:00 n = 60


# 26. Baselines come before sophisticated models

A forecasting model should beat simple alternatives.

We use two baselines.

### Naïve

$$
\hat Y_{t+h|t}=Y_t.
$$

Everything in the future equals the latest observed value.

### Seasonal naïve

For monthly seasonality:

$$
\hat Y_{t+h|t}=Y_{t+h-12}.
$$

This says:

> forecast each month using the corresponding month from the previous year.

Seasonal naïve is surprisingly difficult to beat when seasonality is strong.

In [56]:
def naive_forecast(train: pd.Series, horizon: int, future_index: pd.Index) -> pd.Series:
    return pd.Series(train.iloc[-1], index=future_index, name="Naive")


def seasonal_naive_forecast(
    train: pd.Series,
    horizon: int,
    season_length: int,
    future_index: pd.Index,
) -> pd.Series:
    history = list(train.values)
    preds = []
    for _ in range(horizon):
        pred = history[-season_length]
        preds.append(pred)
        history.append(pred)
    return pd.Series(preds, index=future_index, name="Seasonal naive")


naive_pred = naive_forecast(train, len(test), test.index)
snaive_pred = seasonal_naive_forecast(train, len(test), 12, test.index)

baseline_metrics = pd.DataFrame([
    {"Model": "Naive", **forecast_metrics(test, naive_pred)},
    {"Model": "Seasonal naive", **forecast_metrics(test, snaive_pred)},
])

display(baseline_metrics.round(4))

show(make_time_plot(
    {
        "Actual": test,
        "Naive": naive_pred,
        "Seasonal naive": snaive_pred,
    },
    "Baseline forecasts on the held-out period",
    y_label="CO₂",
))

,Model,MAE,RMSE,MAPE_%
0,Naive,5.5042,6.1881,1.4906
1,Seasonal naive,5.1592,5.7273,1.3986


# 27. Fit Holt-Winters / ETS-style model

Because the CO₂ series has:

- trend;
- approximately stable seasonal amplitude;
- annual seasonality;

an additive trend + additive seasonal model is a reasonable candidate.

This is a modelling hypothesis that we will test empirically—not a fact simply because the plot looks seasonal.

In [57]:
ets_fit = ExponentialSmoothing(
    train,
    trend="add",
    seasonal="add",
    seasonal_periods=12,
    initialization_method="estimated",
).fit(optimized=True)

ets_pred = ets_fit.forecast(len(test))
ets_pred.index = test.index

display(pd.Series({
    "smoothing_level": ets_fit.params.get("smoothing_level"),
    "smoothing_trend": ets_fit.params.get("smoothing_trend"),
    "smoothing_seasonal": ets_fit.params.get("smoothing_seasonal"),
}).round(5))

show(make_time_plot(
    {
        "Actual": test,
        "ETS/Holt-Winters forecast": ets_pred,
    },
    "ETS/Holt-Winters forecast",
    y_label="CO₂",
))

smoothing_level       0.56734
smoothing_trend       0.00000
smoothing_seasonal    0.13487
dtype: float64

# 28. Candidate seasonal ARIMA models

Rather than searching a huge parameter grid blindly, we start with a small interpretable candidate set.

Each candidate has:

$$
D=1,\qquad s=12,
$$

to account for annual seasonal non-stationarity.

We vary the non-seasonal AR and MA orders.

AIC is:

$$
AIC=-2\log L+2k.
$$

Lower AIC rewards fit while penalising unnecessary parameters.

Important:

> AIC compares likelihood-based models fitted to the same response data.  
> It is **not** a replacement for out-of-sample forecast evaluation.

In [58]:
@dataclass(frozen=True)
class SarimaSpec:
    order: Tuple[int, int, int]
    seasonal_order: Tuple[int, int, int, int]

    @property
    def label(self) -> str:
        return f"SARIMA{self.order}x{self.seasonal_order}"


candidate_specs = [
    SarimaSpec((0, 1, 1), (0, 1, 1, 12)),
    SarimaSpec((1, 1, 0), (0, 1, 1, 12)),
    SarimaSpec((1, 1, 1), (0, 1, 1, 12)),
    SarimaSpec((2, 1, 1), (0, 1, 1, 12)),
]

sarima_results = {}
rows = []

for spec in candidate_specs:
    model = SARIMAX(
        train,
        order=spec.order,
        seasonal_order=spec.seasonal_order,
        trend="n",
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    fitted = model.fit(disp=False, maxiter=300)
    sarima_results[spec.label] = fitted

    rows.append({
        "Model": spec.label,
        "AIC": fitted.aic,
        "BIC": fitted.bic,
        "LogLik": fitted.llf,
        "Parameters": len(fitted.params),
    })

candidate_table = (
    pd.DataFrame(rows)
    .sort_values("AIC")
    .reset_index(drop=True)
)

display(candidate_table.round(3))

,Model,AIC,BIC,LogLik,Parameters
0,"SARIMA(1, 1, 1)x(0, 1, 1, 12)",199.715,216.053,-95.858,4
1,"SARIMA(2, 1, 1)x(0, 1, 1, 12)",201.713,222.135,-95.856,5
2,"SARIMA(0, 1, 1)x(0, 1, 1, 12)",202.459,214.712,-98.229,3
3,"SARIMA(1, 1, 0)x(0, 1, 1, 12)",211.412,223.672,-102.706,3


# 29. Select the AIC-best candidate, then test it out-of-sample

Notice the two-stage logic:

### In-sample evidence

Use likelihood/AIC/BIC and residual diagnostics.

### Out-of-sample evidence

Use the held-out future period.

A model can have excellent in-sample likelihood yet mediocre forecasting performance.

Forecasting is ultimately about future behaviour.

In [59]:
best_label = candidate_table.iloc[0]["Model"]
best_sarima_fit = sarima_results[best_label]

sarima_forecast_result = best_sarima_fit.get_forecast(steps=len(test))
sarima_pred = sarima_forecast_result.predicted_mean.copy()
sarima_pred.index = test.index

sarima_ci = sarima_forecast_result.conf_int(alpha=0.05).copy()
sarima_ci.index = test.index

print("Selected:", best_label)

show(make_time_plot(
    {
        "Actual": test,
        "SARIMA forecast": sarima_pred,
    },
    f"Held-out forecast — {best_label}",
    y_label="CO₂",
))

Selected: SARIMA(1, 1, 1)x(0, 1, 1, 12)


# 30. Prediction intervals

A point forecast answers:

$$
E[Y_{t+h}\mid \mathcal F_t].
$$

But a real forecasting system should also quantify uncertainty.

A typical 95% prediction interval has the conceptual form

$$
\hat Y_{t+h|t}
\pm
1.96\,\widehat{\operatorname{SE}}_h.
$$

Forecast uncertainty normally widens with the horizon because more future shocks remain unobserved.

The shaded region below is the model's approximate 95% forecast interval.

In [60]:
source = ColumnDataSource(pd.DataFrame({
    "date": test.index,
    "actual": test.values,
    "forecast": sarima_pred.values,
    "lower": sarima_ci.iloc[:, 0].values,
    "upper": sarima_ci.iloc[:, 1].values,
}))

p = figure(
    width=900,
    height=380,
    x_axis_type="datetime",
    title=f"{best_label} — forecast with 95% prediction interval",
    x_axis_label="Time",
    y_axis_label="CO₂",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

p.line("date", "actual", source=source, line_width=2, legend_label="Actual")
p.line(
    "date",
    "forecast",
    source=source,
    line_width=2,
    line_dash="dashed",
    legend_label="Forecast",
)

band = Band(
    base="date",
    lower="lower",
    upper="upper",
    source=source,
    fill_alpha=0.20,
    line_alpha=0.25,
)
p.add_layout(band)

p.legend.location = "top_left"
show(p)

# 31. Compare all point forecasts

We now compare:

1. naïve;
2. seasonal naïve;
3. Holt-Winters / ETS;
4. selected SARIMA.

Metrics:

$$
MAE=
\frac1n\sum_{t=1}^n|Y_t-\hat Y_t|
$$

$$
RMSE=
\sqrt{
\frac1n
\sum_{t=1}^n(Y_t-\hat Y_t)^2
}
$$

$$
MAPE=
\frac{100}{n}
\sum_{t=1}^n
\left|
\frac{Y_t-\hat Y_t}{Y_t}
\right|.
$$

RMSE penalises large errors more aggressively than MAE.

In [61]:
forecast_map = {
    "Naive": naive_pred,
    "Seasonal naive": snaive_pred,
    "ETS/Holt-Winters": ets_pred,
    best_label: sarima_pred,
}

comparison = pd.DataFrame([
    {"Model": name, **forecast_metrics(test, pred)}
    for name, pred in forecast_map.items()
]).sort_values("RMSE").reset_index(drop=True)

display(comparison.round(4))

,Model,MAE,RMSE,MAPE_%
0,"SARIMA(1, 1, 1)x(0, 1, 1, 12)",0.8408,0.9363,0.2285
1,ETS/Holt-Winters,1.5629,1.7280,0.4240
2,Seasonal naive,5.1592,5.7273,1.3986
3,Naive,5.5042,6.1881,1.4906


In [62]:
show(make_time_plot(
    {
        "Actual": test,
        "Seasonal naive": snaive_pred,
        "ETS/Holt-Winters": ets_pred,
        "SARIMA": sarima_pred,
    },
    "Forecast comparison on the same held-out future",
    y_label="CO₂",
))

# Part VII — Residual diagnostics

# 32. A fitted model is not accepted merely because it converged

Define residuals:

$$
e_t=Y_t-\hat Y_t.
$$

If the model has captured the systematic temporal dependence, the residuals should be approximately unpredictable.

The ideal target is **white noise**:

$$
E[e_t]\approx0,
$$

$$
\operatorname{Var}(e_t)\approx\sigma^2,
$$

and

$$
\operatorname{Corr}(e_t,e_{t-k})\approx0
\quad
\text{for }k>0.
$$

We therefore inspect:

- residual time plot;
- residual histogram;
- residual ACF;
- Ljung–Box test.

In [63]:
resid = pd.Series(best_sarima_fit.resid).dropna()
# Remove the early diffuse-initialisation portion from visual diagnostics.
resid_diag = resid.iloc[24:].copy()

show(column(
    make_time_plot(
        {"Residual": resid_diag},
        f"{best_label} — residuals over time",
        y_label="Residual",
    ),
    make_histogram(
        resid_diag,
        f"{best_label} — residual distribution",
        bins=35,
    ),
    make_acf_plot(
        resid_diag,
        f"{best_label} — residual ACF",
        nlags=36,
    ),
))

# 33. Ljung–Box test

For a selected lag horizon $m$, the Ljung–Box null hypothesis is approximately

$$
H_0:
\rho_1=\rho_2=\cdots=\rho_m=0.
$$

If the p-value is very small, the residuals still contain statistically detectable serial correlation.

That suggests the model may have failed to capture some temporal structure.

However:

- a large sample can detect tiny, practically irrelevant correlations;
- a high p-value does not prove the model is correct;
- diagnostics should be interpreted together.

In [33]:
lb = acorr_ljungbox(
    resid_diag,
    lags=[12, 24, 36],
    return_df=True,
)

display(lb.round(4))

,lb_stat,lb_pvalue
12,9.9916,0.6167
24,18.0681,0.7997
36,28.0365,0.8259


# 34. Residual diagnostics versus forecast accuracy

These answer different questions.

### Diagnostics

> Is the assumed stochastic model a plausible description of the remaining dependence?

### Forecast accuracy

> Does the model predict unseen future observations well?

A model can forecast well while still being structurally imperfect.

Another model can have textbook residual diagnostics yet lose to a simple seasonal naïve benchmark.

Good applied time-series work checks **both**.

# Part VIII — Rolling-origin evaluation

# 35. Why one train/test split may be insufficient

A single holdout asks:

> How did the model perform during one specific historical future?

But forecast quality can vary by period.

Rolling-origin evaluation repeatedly simulates the real forecasting process:

```text
Origin 1:  [----------- train -----------] -> next h months

Origin 2:  [--------------- train ---------------] -> next h months

Origin 3:  [------------------- train -------------------] -> next h months
```

This preserves causality:

$$
\text{training information always occurs before evaluation information}.
$$

To keep the notebook reasonably fast, we use a small number of six-month forecast origins.

In [64]:
def rolling_origin_evaluate(
    series: pd.Series,
    spec: SarimaSpec,
    horizon: int = 6,
    n_origins: int = 6,
    step: int = 6,
) -> pd.DataFrame:
    rows = []
    n = len(series)

    first_train_end = n - horizon - step * (n_origins - 1)

    for origin_id in range(n_origins):
        train_end = first_train_end + origin_id * step
        tr = series.iloc[:train_end]
        te = series.iloc[train_end: train_end + horizon]

        if len(te) < horizon:
            continue

        # Seasonal-naive baseline
        sn = seasonal_naive_forecast(tr, horizon, 12, te.index)
        rows.append({
            "origin": origin_id + 1,
            "train_end": tr.index[-1],
            "model": "Seasonal naive",
            **forecast_metrics(te, sn),
        })

        # SARIMA
        fitted = SARIMAX(
            tr,
            order=spec.order,
            seasonal_order=spec.seasonal_order,
            trend="n",
            enforce_stationarity=False,
            enforce_invertibility=False,
        ).fit(disp=False, maxiter=200)

        pred = fitted.get_forecast(horizon).predicted_mean
        pred.index = te.index

        rows.append({
            "origin": origin_id + 1,
            "train_end": tr.index[-1],
            "model": "SARIMA",
            **forecast_metrics(te, pred),
        })

    return pd.DataFrame(rows)


best_spec = next(spec for spec in candidate_specs if spec.label == best_label)

rolling_results = rolling_origin_evaluate(
    co2_monthly,
    best_spec,
    horizon=6,
    n_origins=6,
    step=6,
)

display(rolling_results.round(4))

,origin,train_end,model,MAE,RMSE,MAPE_%
0,1,1998-12-01,Seasonal naive,2.2175,2.2673,0.5999
1,1,1998-12-01,SARIMA,0.3162,0.4367,0.0854
2,2,1999-06-01,Seasonal naive,1.0042,1.0260,0.2737
3,2,1999-06-01,SARIMA,0.1956,0.2338,0.0534
4,3,1999-12-01,Seasonal naive,0.8292,0.8748,0.2237
5,3,1999-12-01,SARIMA,0.3983,0.4735,0.1074
6,4,2000-06-01,Seasonal naive,1.4425,1.4710,0.3921
7,4,2000-06-01,SARIMA,0.2407,0.2710,0.0654
8,5,2000-12-01,Seasonal naive,1.5975,1.6380,0.4291
9,5,2000-12-01,SARIMA,0.2749,0.3562,0.0739


In [65]:
rolling_summary = (
    rolling_results
    .groupby("model")[["MAE", "RMSE", "MAPE_%"]]
    .agg(["mean", "std"])
)

display(rolling_summary.round(4))

MAE            RMSE          MAPE_%        
                  mean     std    mean     std    mean     std
model                                                         
SARIMA          0.2588  0.0945  0.3224  0.1208  0.0700  0.0254
Seasonal naive  1.4132  0.4880  1.4450  0.4935  0.3823  0.1317

In [66]:
def metric_by_origin_plot(df, metric="RMSE"):
    p = figure(
        width=850,
        height=330,
        title=f"Rolling-origin {metric}",
        x_axis_label="Forecast origin",
        y_axis_label=metric,
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )

    dashes = ["solid", "dashed", "dotted"]
    for i, (model, part) in enumerate(df.groupby("model")):
        p.line(
            part["origin"],
            part[metric],
            line_width=2,
            line_dash=dashes[i % len(dashes)],
            legend_label=model,
        )
        p.scatter(
            part["origin"],
            part[metric],
            size=7,
            legend_label=model,
        )

    p.legend.location = "top_left"
    p.legend.click_policy = "hide"
    return p


show(metric_by_origin_plot(rolling_results, "RMSE"))

## Why this evaluation is stronger

Suppose one model wins at a single test split by chance.

Rolling-origin evaluation asks whether the advantage is **stable across multiple historical forecast origins**.

That is much closer to the operational question:

> If I repeatedly deployed this model through time, how reliably would it perform?

# Part IX — Refit on all available data and forecast the future

# 36. Final forecasting workflow

After model selection and validation, a common production workflow is:

1. choose the model using historical validation;
2. refit that chosen specification on **all available observations**;
3. forecast genuinely unseen future periods.

We now refit the chosen SARIMA specification to the complete monthly CO₂ history and forecast 24 months.

In [67]:
final_model = SARIMAX(
    co2_monthly,
    order=best_spec.order,
    seasonal_order=best_spec.seasonal_order,
    trend="n",
    enforce_stationarity=False,
    enforce_invertibility=False,
)

final_fit = final_model.fit(disp=False, maxiter=300)

HORIZON = 24
future_result = final_fit.get_forecast(HORIZON)
future_mean = future_result.predicted_mean
future_ci = future_result.conf_int(alpha=0.05)

display(pd.DataFrame({
    "forecast": future_mean,
    "lower_95": future_ci.iloc[:, 0],
    "upper_95": future_ci.iloc[:, 1],
}).head(12).round(3))

,forecast,lower_95,upper_95
2002-01-01,371.979,371.394,372.564
2002-02-01,372.760,372.054,373.466
2002-03-01,373.671,372.882,374.460
2002-04-01,374.859,373.999,375.718
2002-05-01,375.341,374.417,376.265
2002-06-01,374.768,373.784,375.752
2002-07-01,373.248,372.207,374.289
2002-08-01,371.217,370.122,372.312
2002-09-01,369.505,368.358,370.651
2002-10-01,369.733,368.537,370.928


In [69]:
history_tail = co2_monthly.iloc[-120:]

future_df = pd.DataFrame({
    "date": future_mean.index,
    "forecast": future_mean.values,
    "lower": future_ci.iloc[:, 0].values,
    "upper": future_ci.iloc[:, 1].values,
})
future_source = ColumnDataSource(future_df)

p = figure(
    width=950,
    height=400,
    x_axis_type="datetime",
    title=f"24-month future forecast — {best_spec.label}",
    x_axis_label="Time",
    y_axis_label="CO₂",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

p.line(
    history_tail.index,
    history_tail.values,
    line_width=2,
    legend_label="Recent history",
)
p.line(
    "date",
    "forecast",
    source=future_source,
    line_width=2,
    line_dash="dashed",
    legend_label="Forecast",
)

future_band = Band(
    base="date",
    lower="lower",
    upper="upper",
    source=future_source,
    fill_alpha=0.20,
    line_alpha=0.25,
)
p.add_layout(future_band)

p.legend.location = "top_left"
show(p)

# Part X — Connecting the theory

# 37. Stationarity of AR models

For AR(1):

$$
Y_t=\phi Y_{t-1}+\varepsilon_t.
$$

Repeated substitution gives

$$
Y_t
=
\varepsilon_t
+
\phi\varepsilon_{t-1}
+
\phi^2\varepsilon_{t-2}
+\cdots.
$$

If

$$
|\phi|<1,
$$

then

$$
\phi^k\rightarrow0.
$$

Old shocks eventually disappear.

For AR($p$),

$$
\phi(B)Y_t=\varepsilon_t
$$

with

$$
\phi(z)=1-\phi_1z-\cdots-\phi_pz^p.
$$

Stationarity requires the roots of $\phi(z)=0$ to lie outside the unit circle.

This is the higher-order version of the AR(1) requirement $|\phi|<1$.

# 38. Invertibility of MA models

For MA(1):

$$
Y_t=\varepsilon_t+\theta\varepsilon_{t-1}.
$$

Invertibility ensures that the unobserved innovations can be represented uniquely in terms of present and past observations.

Under a common sign convention, MA(1) is invertible when

$$
|\theta|<1.
$$

Why do we care?

Without invertibility, different parameterisations can imply the same autocorrelation structure, making the model representation non-unique.

So:

- **stationarity** stabilises AR dynamics;
- **invertibility** stabilises the MA representation.

# 39. ACF/PACF should guide—not dictate—model order

A simplistic rule says:

> PACF cutoff at 2 → fit AR(2).

Real data are rarely that clean.

Reasons include:

- finite-sample noise;
- trend or seasonality not completely removed;
- AR and MA effects interacting;
- near parameter cancellation;
- structural changes;
- measurement noise.

A better workflow is:

$$
\boxed{\text{ACF/PACF}}
\rightarrow
\boxed{\text{small candidate set}}
\rightarrow
\boxed{\text{AIC/BIC}}
\rightarrow
\boxed{\text{residual diagnostics}}
\rightarrow
\boxed{\text{out-of-sample forecasting}}
$$

# 40. Why forecast intervals widen

Suppose an AR(1) is

$$
Y_t=\phi Y_{t-1}+\varepsilon_t.
$$

One-step future:

$$
Y_{t+1}
=
\phi Y_t+\varepsilon_{t+1}.
$$

Only one future shock is unknown.

Two steps:

$$
Y_{t+2}
=
\phi^2Y_t
+
\phi\varepsilon_{t+1}
+
\varepsilon_{t+2}.
$$

Now two future shocks are unknown.

As the horizon increases, forecast uncertainty accumulates.

This is why point forecasts become less informative farther into the future and why prediction intervals are indispensable.

# Part XI — Practical modelling decision tree

# 41. A methodical ST4253 workflow

When handed a new time series, use the following reasoning sequence.

### Step 1 — Clarify the data-generating context

Ask:

- What does one observation represent?
- What is the sampling frequency?
- Are timestamps equally spaced?
- Are there missing periods?
- Is the target a stock or a flow?
- Are revisions possible?

### Step 2 — Plot the raw series

Look for:

- trend;
- seasonality;
- changing variance;
- outliers;
- structural breaks;
- unusual periods.

### Step 3 — Establish baselines

At minimum consider:

- naïve;
- seasonal naïve.

### Step 4 — Transform/decompose if useful

Possible tools:

- logarithm / Box-Cox-style transformation;
- STL decomposition;
- detrending;
- differencing.

### Step 5 — Investigate stationarity

Combine:

- time plots;
- rolling summaries;
- ACF;
- domain understanding;
- ADF-type tests.

### Step 6 — Inspect ACF/PACF

Use them to build a **small plausible model set**.

### Step 7 — Fit candidate models

Examples:

- ETS;
- ARIMA;
- seasonal ARIMA.

### Step 8 — Compare in-sample evidence

Consider:

- AIC;
- BIC;
- parameter plausibility.

### Step 9 — Diagnose residuals

Check:

- residual plot;
- ACF;
- Ljung–Box;
- unusual shocks;
- changing residual variance.

### Step 10 — Backtest chronologically

Use:

- final holdout;
- preferably rolling-origin validation.

### Step 11 — Quantify uncertainty

Report prediction intervals, not only point predictions.

### Step 12 — Refit and deploy

Once a specification is selected, refit it on all currently available observations and forecast forward.

# Part XII — Common mistakes and how to avoid them

# 42. Mistake: random cross-validation

Wrong:

```python
train_test_split(X, y, shuffle=True)
```

for genuinely ordered forecasting data.

Why?

Future observations can enter the training set while earlier observations appear in validation.

That creates temporal leakage.

---

# 43. Mistake: choosing ARIMA solely from ACF/PACF

ACF/PACF are identification aids, not an oracle.

Always check:

- likelihood criteria;
- residual diagnostics;
- forecast accuracy.

---

# 44. Mistake: selecting the smallest AIC and stopping

AIC measures relative in-sample information loss.

It does not guarantee the best future RMSE.

Use historical forecast validation.

---

# 45. Mistake: ignoring simple baselines

If a sophisticated model cannot beat seasonal naïve, the sophisticated model has not earned its complexity.

---

# 46. Mistake: demanding stationary raw data

Many useful economic, environmental and business series are non-stationary.

The task is to model their structure appropriately—not discard them.

---

# 47. Mistake: over-differencing

Differencing until the series "looks random" can destroy useful information.

Use the minimum differencing justified by theory, diagnostics and forecasting performance.

---

# 48. Mistake: reporting only point forecasts

A prediction such as

$$
\hat Y_{t+12}=430
$$

without an uncertainty interval creates false precision.

Forecast risk increases with horizon and should be communicated.

# Part XIII — Exercises

Try these before reading the solution discussion.

## Exercise 1 — AR(1) persistence

Simulate

$$
Y_t=\phi Y_{t-1}+\varepsilon_t
$$

for

$$
\phi\in\{0.2,0.6,0.95,-0.7\}.
$$

For each process:

1. plot the series;
2. plot the ACF;
3. compare how quickly dependence disappears;
4. explain the oscillating ACF when $\phi<0$.

---

## Exercise 2 — Unit root

Simulate:

$$
Y_t=0.98Y_{t-1}+\varepsilon_t
$$

and

$$
Z_t=Z_{t-1}+\varepsilon_t.
$$

Can a highly persistent stationary process visually resemble a unit-root process in a finite sample?

What does this tell you about relying on plots alone?

---

## Exercise 3 — Seasonal differencing

Generate monthly data with:

- linear trend;
- annual seasonality;
- Gaussian noise.

Compare:

$$
\Delta Y_t
$$

with

$$
\Delta_{12}Y_t.
$$

Which component does each transformation target?

---

## Exercise 4 — Forecast competition

For the CO₂ case study:

1. change the test window from 60 to 36 months;
2. repeat the model comparison;
3. determine whether the model ranking changes.

This illustrates that forecast performance can depend on the evaluation period.

---

## Exercise 5 — SARIMA sensitivity

Add candidate models such as:

$$
(1,1,2)(0,1,1)_{12}
$$

and

$$
(1,1,1)(1,1,0)_{12}.
$$

Compare:

- AIC;
- Ljung–Box results;
- test RMSE.

Does the AIC winner always have the lowest test RMSE?

# 49. Exercise solution intuition

## Exercise 1

For AR(1),

$$
\rho_k=\phi^k.
$$

Therefore:

- $\phi=0.2$: very rapid decay;
- $\phi=0.6$: moderate persistence;
- $\phi=0.95$: very slow decay;
- $\phi=-0.7$: alternating signs because $(-0.7)^k$ alternates.

---

## Exercise 2

Yes. A near-unit-root stationary process can look strikingly similar to a random walk over a finite sample.

This is one reason unit-root testing is statistically difficult.

---

## Exercise 3

Ordinary first differencing mainly attacks slowly changing level/trend.

Seasonal differencing directly attacks repeated lag-$s$ structure.

Sometimes both are required.

---

## Exercise 4

The ranking may change.

A single holdout period samples only one historical regime.  
This motivates rolling-origin evaluation.

---

## Exercise 5

Not necessarily.

AIC estimates relative expected information loss under model assumptions; test RMSE measures realised predictive error on a particular future sample.

They answer related but different questions.

# Part XIV — What ST4253 gives a data scientist

# 50. Classical forecasting ideas that remain valuable in modern ML

Even if you later use:

- XGBoost with lag features;
- Random Forests;
- temporal convolution networks;
- LSTMs/GRUs;
- Transformers;
- foundation models for time series;

the classical ideas remain essential.

### Temporal leakage

You must respect chronological ordering.

### Baselines

Seasonal naïve and ETS can be very strong.

### Residual structure

Prediction errors should be investigated over time, not only averaged into one metric.

### Seasonality

Calendar structure should be modelled explicitly.

### Forecast horizon

A model good at $h=1$ may be poor at $h=24$.

### Uncertainty

Forecasting is fundamentally probabilistic.

### Non-stationarity

Distribution shift through time is not an edge case—it is the normal setting.

# 51. Final synthesis

The core conceptual chain of ST4253 can be summarised as:

$$
\boxed{
\text{time dependence}
\rightarrow
\text{ACF/PACF}
\rightarrow
\text{stationarity}
\rightarrow
\text{AR/MA}
\rightarrow
\text{ARMA}
\rightarrow
\text{differencing}
\rightarrow
\text{ARIMA}
}
$$

alongside the forecasting-components chain:

$$
\boxed{
\text{level}
\rightarrow
\text{trend}
\rightarrow
\text{seasonality}
\rightarrow
\text{exponential smoothing / ETS}
}
$$

These come together through:

$$
\boxed{
\text{model selection}
+
\text{diagnostics}
+
\text{chronological validation}
+
\text{forecast uncertainty}
}
$$

The most important habit is not memorising model names.

It is learning to ask, in order:

> **What temporal structure exists? What assumptions does my model make? What structure remains unexplained? And does the model actually forecast unseen future data better than a simple baseline?**

That reasoning pattern is the durable skill behind applied time-series analysis.

# 52. Suggested extensions for a second notebook

Once the classical ST4253 foundation is comfortable, natural extensions include:

- dynamic regression / ARIMAX;
- intervention analysis;
- structural breaks;
- state-space models and Kalman filtering;
- stochastic volatility;
- ARCH/GARCH;
- VAR for multivariate series;
- cointegration and error-correction models;
- hierarchical forecasting;
- probabilistic forecast scoring;
- conformal prediction for time series;
- gradient-boosted trees with lag features;
- Random Forest forecasting;
- recurrent neural networks;
- temporal convolutional networks;
- Transformers;
- modern time-series foundation models.

A useful progression is:

$$
\boxed{\text{Classical stochastic structure}}
\rightarrow
\boxed{\text{state-space / multivariate}}
\rightarrow
\boxed{\text{ML forecasting}}
\rightarrow
\boxed{\text{deep probabilistic forecasting}}
$$

The classical material should come first because it teaches what temporal dependence, seasonality, leakage, uncertainty and forecast evaluation actually mean.